In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

pd.set_option("display.max_columns", 25)
df = pd.read_csv("../data/raw/store-data.csv")

NumPy: 1.26.4
Pandas: 3.0.5


## delete duplicate rows

In [2]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())

duplicate_row_ids = df[df['Row ID'].duplicated(keep=False)].sort_values('Row ID')

display(duplicate_row_ids)

print(df.loc[df['Row ID'].duplicated(keep=False), 'Row ID'].nunique())

df = df.drop_duplicates("Row ID")

Shape: (10064, 21)

Columns:
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']

Data types:
Row ID             int64
Order ID             str
Order Date           str
Ship Date            str
Ship Mode            str
Customer ID          str
Customer Name        str
Segment              str
Country              str
City                 str
State                str
Postal Code          str
Region               str
Product ID           str
Category             str
Sub-Category         str
Product Name         str
Sales            float64
Quantity         float64
Discount         float64
Profit           float64
dtype: object

Missing values:
Row ID             0
Order ID           0
Order Date         0
Ship Date        101
Ship Mode        301
Customer ID        0
Customer Name   

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
35,36,CA-2016-117590,12/8/2016,12/10/2016,First Class,GH-14485,NaN,Corporate,United States,Richardson,Texas,75080.0,Central,TEC-PH-10004977,Technology,Phones,GE 30524EE4,1097.544,7.0,0.2,123.4737
10030,36,CA-2016-117590,12/8/2016,12/10/2016,First Class,GH-14485,NaN,Corporate,United States,Richardson,Texas,75080.0,Central,TEC-PH-10004977,Technology,Phones,GE 30524EE4,1097.544,7.0,0.2,123.4737
39,40,CA-2015-117415,12/27/2015,12/31/2015,Standard Class,SN-20710,NaN,Home Office,United States,Houston,Texas,77041.0,Central,FUR-CH-10004218,Furniture,Chairs,"Global Fabric Manager's Chair, Dark Gray",212.058,3.0,0.3,-15.1470
9997,40,CA-2015-117415,12/27/2015,12/31/2015,Standard Class,SN-20710,NaN,Home Office,United States,Houston,Texas,77041.0,Central,FUR-CH-10004218,Furniture,Chairs,"Global Fabric Manager's Chair, Dark Gray",212.058,3.0,0.3,-15.1470
65,66,CA-2015-135545,11/24/2015,11/30/2015,Standard Class,KM-16720,Kunst Miller,Consumer,United States,Los Angeles,California,90004.0,West,FUR-FU-10000397,Furniture,Furnishings,Luxo Economy Swing Arm Lamp,79.760,4.0,0.0,22.3328
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9404,9405,CA-2017-141663,4/13/2017,4/17/2017,Standard Class,DP-13105,NaN,Corporate,United States,Philadelphia,Pennsylvania,19134.0,East,OFF-FA-10004076,Office Supplies,Fasteners,Translucent Push Pins by OIC,7.920,5.0,0.2,1.6830
10001,9952,CA-2017-121559,6/1/2017,6/3/2017,Second Class,HW-14935,NaN,Corporate,United States,Indianapolis,Indiana,46203.0,Central,OFF-BI-10002072,Office Supplies,Binders,Cardinal Slant-D Ring Binders,17.380,2.0,0.0,8.6900
9951,9952,CA-2017-121559,6/1/2017,6/3/2017,Second Class,HW-14935,NaN,Corporate,United States,Indianapolis,Indiana,46203.0,Central,OFF-BI-10002072,Office Supplies,Binders,Cardinal Slant-D Ring Binders,17.380,2.0,0.0,8.6900
10053,9984,US-2016-157728,9/22/2016,9/28/2016,Standard Class,RC-19960,NaN,Consumer,United States,Grand Rapids,Michigan,49505.0,Central,TEC-PH-10001305,Technology,Phones,Panasonic KX TS208W Corded phone,97.980,2.0,0.0,27.4344


70


## Data types

In [3]:
## change the data formate

df['Order Date'] = pd.to_datetime(
    df['Order Date'],
    format='mixed',
    errors='coerce'
)

df['Ship Date'] = pd.to_datetime(
    df['Ship Date'],
    format='mixed',
    errors='coerce'
)

df[['Order Date', 'Ship Date']].dtypes

Order Date    datetime64[us]
Ship Date     datetime64[us]
dtype: object

In [4]:
# Fill Ship Mode and Ship Date using the same Order ID
for col in ['Ship Mode', 'Ship Date']:
    df[col] = (
        df.groupby('Order ID')[col]
        .transform(lambda x: x.ffill().bfill())
    )

# Calculate actual shipping days
df['Shipping Days'] = (
    df['Ship Date'] - df['Order Date']
).dt.days

# Calculate typical shipping time for each Ship Mode
estimated_days = df.groupby('Ship Mode')['Shipping Days'].median()

# Assign the estimated number of days based on Ship Mode
df['Estimated Days'] = df['Ship Mode'].map(estimated_days)

# Fill missing Ship Date

df['Ship Date'] = df['Ship Date'].fillna(
    df['Order Date'] + pd.to_timedelta(df['Estimated Days'], unit='D')
)

# Recalculate Shipping Days AFTER filling Ship Date
df['Shipping Days'] = (
    df['Ship Date'] - df['Order Date']
).dt.days

# Estimate missing Ship Mode from Shipping Days
def estimate_ship_mode(days):
    if pd.isna(days):
        return pd.NA
    return (estimated_days - days).abs().idxmin()

df['Ship Mode'] = df['Ship Mode'].fillna(
    df['Shipping Days'].apply(estimate_ship_mode)
)

# Update the estimated days
df['Estimated Days'] = df['Ship Mode'].map(estimated_days)


## Customer and Segment

In [5]:
# Fill Customer Name using the same Customer ID

df['Customer Name'] = df.groupby('Customer ID')['Customer Name'].transform(lambda x: x.ffill().bfill())

# Fill remaining missing names
df['Customer Name'] = df['Customer Name'].fillna('Unknown')

# normalize names
df['Customer Name'] = (
    df['Customer Name']
    .str.lower()
    .str.split()
    .apply(lambda x: ' '.join(sorted(x)) if isinstance(x, list) else x)
)

segment_mapping = {
    'Home Ofice': 'Home Office',
    'Consumerr': 'Consumer',
    'Corporrate': 'Corporate'
}

df['Segment'] = df['Segment'].replace(segment_mapping)

print("Missing Customer Name:")
print(df['Customer Name'].isna().sum())

print("\nSegment values:")
print(df['Segment'].value_counts(dropna=False))

Missing Customer Name:
0

Segment values:
Segment
Consumer       5191
Corporate      3020
Home Office    1783
Name: count, dtype: int64


## City, State, Postal Code and Region

In [6]:
# titled the cites
df['City'] = df['City'].str.strip().str.title()
# states was good
# post code
df['Postal Code'] = df.groupby(['City', 'State'])['Postal Code'].transform(lambda x: x.ffill().bfill())
# region was good
print("Postal Code:")
print(df['Postal Code'].isna().sum())

Postal Code:
1


## Product, Category and sub_category

In [7]:
# correct the product name with the most frequent one

df['Product Name'] = (
    df.groupby('Product ID')['Product Name'].transform(lambda x: x.mode()[0] if not x.mode().empty else x)
)
# Category
# the category name
df['Category'] = df['Category'].str.strip().str.title()

# sub_category was good
print("Product Name : ")
print(df['Product Name'].isna().sum())


Product Name : 
0


## sales, Quantity, Discount and Profite

In [8]:
# sales = Price * (1 - Discount) * Quantity
# product_price = sales / ( (1 - Discount) * Quantity )
# also we need product cost

# Remove invalid values BEFORE calculations

df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['Discount'] = pd.to_numeric(df['Discount'], errors='coerce')
df['Sales'] = pd.to_numeric(df['Sales'], errors='coerce')
df['Profit'] = pd.to_numeric(df['Profit'], errors='coerce')

df = df.drop(
    df[(df['Quantity'] <= 0) | pd.isna(df['Quantity'])].index
)

df = df.drop(
    df[(df['Discount'] < 0) | (df['Discount'] > 1) | pd.isna(df['Discount'])].index
)

def _mode(x):
    m = x.dropna().mode()
    return m.iloc[0] if len(m) else float('nan')

df['Price'] = df['Sales'] / ((1 - df['Discount']) * df['Quantity'])
df['Price'] = df['Price'].replace([np.inf, -np.inf], np.nan)
# fix the prices
df['Price'] = df['Price'].fillna(df.groupby('Product ID')['Price'].transform(_mode))
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

# fix the sales
df['Sales'] = df['Price'] * (1 - df['Discount']) * df['Quantity']

# fix the quantity
df['Quantity'] = (df['Sales'] / ((1 - df['Discount']) * df['Price']).replace(0, np.nan).replace([np.inf, -np.inf], np.nan)).round().astype('Int64')

# we will fix the quantity and sales trough profit
# fixing the profit
# profit = sales - (product cost * Quantity)
# product cost = (sales - profit) / Quantity

df['Product Cost'] = (df['Sales'] - df['Profit']) / df['Quantity']
df['Product Cost'] = df['Product Cost'].replace([np.inf, -np.inf], np.nan)

# fix the Product cost
df['Product Cost'] = df.groupby('Product ID')['Product Cost'].transform(_mode)
df['Product Cost'] = pd.to_numeric(df['Product Cost'], errors='coerce')

# fix the profit

df['Profit'] = df['Sales'] - (df['Product Cost'] * df['Quantity'])

# the ones who have the same profit ad the same discount must be have the same quantity and the same price
cols = ['Product ID', df['Discount'].round(4), df['Profit'].round(4)]

for col in ['Sales', 'Quantity']:
    fill = df.groupby(cols)[col].transform('first')
    df[col] = df[col].fillna(fill)

# calculate Quantity from Profit
denom = df['Price'] * (1 - df['Discount']) - df['Product Cost']
df['Quantity'] = (df['Profit'] / denom).replace([np.inf, -np.inf], np.nan).round().astype('Int64').fillna(df['Quantity'])
# calculate Sales from Quantity
df['Sales'] = df['Price'] * (1 - df['Discount']) * df['Quantity']

# calcul the quantity and price related to subb_categorty
for col in ['Price', 'Quantity']:
    df[col] = df[col].astype('Float64').fillna(df.groupby('Sub-Category')[col].transform('median'))
df['Quantity'] = df['Quantity'].round().astype('Int64')


# recalculate Sales
df['Sales'] = df['Price'] * (1 - df['Discount']) * df['Quantity']


# the product cost
df['Product Cost'] = (df['Sales'] - df['Profit']) / df['Quantity']


## save date

In [10]:
df.to_csv("./processed/data_cleaned.csv")